# Two-Stage Road Sign Detection

This notebook builds the two-stage pipeline:

1. Binary YOLO detector: every annotated sign, including `other_sign`, becomes class `sign`.
2. MobileNetV3 crop classifier: only the 13 wanted classes are trained.
3. Reject rule: low-confidence classifier outputs are ignored instead of shown as final signs.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "road_signboard_detection_dataset"
TWOSTAGE_DIR = PROJECT_ROOT / "road_sign_twostage_dataset"
DETECTOR_DATA = TWOSTAGE_DIR / "detector_yolo" / "data.yaml"
print("Raw dataset:", RAW_DIR)
print("Two-stage dataset:", TWOSTAGE_DIR)


## 1. Prepare Dataset

This consumes all 9062 XML/JPG pairs, writes a binary YOLO dataset, creates 13-class classifier crops, augments rare classifier classes to at least 300 training crops, and saves reports.

In [ ]:
!python prepare_twostage_dataset.py --recreate


In [ ]:
import json

report = json.loads((TWOSTAGE_DIR / "reports" / "prep_report.json").read_text(encoding="utf-8"))
print("Raw object counts:")
for name, count in report["raw_object_counts"].items():
    print(f"  {name}: {count}")
print("\nSplit image counts:", {k: v["images"] for k, v in report["split_summary"].items()})
print("Related group leakage:", report["related_group_split_leakage"])
print("Detector yaml:", report["detector"]["data_yaml"])


## 2. Train Binary Detectors

Train the Pi candidate first. The reference model is slower but useful for measuring the accuracy ceiling.

In [ ]:
# Pi candidate: smaller detector for Raspberry Pi 4B
!python train_twostage_detector.py --candidate pi --epochs 150 --patience 30 --exist-ok


In [ ]:
# Accuracy reference: larger input/model for comparison
!python train_twostage_detector.py --candidate reference --epochs 150 --patience 30 --exist-ok


## 3. Train And Calibrate Classifier

The classifier reads the crop metadata, uses weighted cross entropy, evaluates macro F1, and calibrates the reject threshold using validation known crops plus held-out `other_sign` crops.

In [ ]:
!python train_twostage_classifier.py --arch mobilenet_v3_large --input-size 224 --epochs 40 --batch-size 64 --export-onnx


In [ ]:
from pathlib import Path

classifier_runs = sorted((PROJECT_ROOT / "runs" / "twostage_classifier").glob("*/summary.json"))
latest_summary = classifier_runs[-1]
summary = json.loads(latest_summary.read_text(encoding="utf-8"))
CLASSIFIER_BEST = Path(summary["best_checkpoint"])
print("Latest classifier:", CLASSIFIER_BEST)
print("Best val macro F1:", summary["best_val_macro_f1"])
print("Reject threshold:", summary["calibration"]["threshold"])


## 4. Two-Stage Inference

Set `SOURCE` to an image, image folder, video, or webcam index (`0`). The final output only draws accepted 13-class signs.

In [ ]:
PI_DETECTOR_BEST = PROJECT_ROOT / "runs" / "twostage_detector" / "yolo26n_sign_640" / "weights" / "best.pt"
SOURCE = TWOSTAGE_DIR / "detector_yolo" / "images" / "test"

!python twostage_infer.py --detector "{PI_DETECTOR_BEST}" --classifier "{CLASSIFIER_BEST}" --source "{SOURCE}" --det-imgsz 640 --det-conf 0.20


## 5. Export Detector For Raspberry Pi

NCNN is usually a strong option on Raspberry Pi. ONNX is also useful for benchmarking and conversion.

In [ ]:
from ultralytics import YOLO

model = YOLO(str(PI_DETECTOR_BEST))
model.export(format="ncnn", imgsz=640)
model.export(format="onnx", imgsz=640, simplify=True)
